In [ ]:
import os
import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import io, transforms

from cbir.models import QueryFeatureExtractor, GeM
from cbir.search import batched_coattention_search_pruned

In [ ]:
FEATURES_PATH = "/home/ubuntu/data/feature_cache/roxford5k_features_pruned.pkl"
IMAGE_PATH = "/home/ubuntu/main-project/test.jpg"
DB_BASE_DIR = "/home/ubuntu/data/datasets/roxford5k/jpg"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
feature_cache = torch.load(FEATURES_PATH)
db_image_ids = list(feature_cache.keys())

db_clusters = torch.stack([v['clusters'] for v in feature_cache.values()], dim=0).to(device)
db_centroids = torch.stack([v['centroid'] for v in feature_cache.values()], dim=0).to(device)
db_radii = torch.stack([v['radius'] for v in feature_cache.values()], dim=0).to(device)

feature_extractor = QueryFeatureExtractor().to(device)
feature_extractor.eval()
gem = GeM(p=3).to(device)

In [ ]:
image = transform(io.read_image(IMAGE_PATH)).unsqueeze(0).to(device)

with torch.no_grad():
    V_q = feature_extractor(image)
    V_q = gem(V_q)
    
    candidate_indices, scores = batched_coattention_search_pruned(
        V_q, db_clusters, db_centroids, db_radii, T=10.0, keep_top_n=100
    )
    
    top_survivor_indices = torch.argsort(scores, descending=True)[:5]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, survivor_idx in enumerate(top_survivor_indices):
    original_db_idx = candidate_indices[survivor_idx]
    image_id = db_image_ids[original_db_idx]
    
    image_path = os.path.join(DB_BASE_DIR, image_id)
    img = Image.open(f"{image_path}.jpg")
    
    axes[i].imshow(img)
    axes[i].set_title(f"Rank {i+1}\nScore: {scores[survivor_idx]:.3f}")
    axes[i].axis('off')

plt.show()